In [20]:
import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.datasets import make_friedman1
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

In [21]:
df = pd.read_csv('Week3_GA_dataset.csv')
df.head()

,V1,V2,V3,V4,V5,Target
0,2.0,50.0,12500.0,98.0,NEGATIVE,YES
1,0.0,13.0,3250.0,28.0,NEGATIVE,YES
2,?,?,4000.0,35.0,NEGATIVE,YES
3,?,20.0,5000.0,45.0,NEGATIVE,YES
4,1.0,24.0,6000.0,77.0,NEGATIVE,NO


Question 1

In [22]:
X = df[['V1', 'V2', 'V3', 'V4', 'V5']]
y = df['Target']
# X.iloc[:, [0,1]]
X.loc[:, ['V1','V2']]

,V1,V2
0,2.0,50.0
1,0.0,13.0
2,?,?
3,?,20.0
4,1.0,24.0
...,...,...
743,23.0,2.0
744,21.0,2.0
745,23.0,3.0
746,39.0,1.0


In [23]:
X = X.copy()

In [24]:
X = X.replace('?', np.nan)

In [25]:
X['V1'] = pd.to_numeric(X['V1'], errors='coerce')
X['V2'] = pd.to_numeric(X['V2'], errors='coerce')
X['V3'] = pd.to_numeric(X['V3'], errors='coerce')
X['V4'] = pd.to_numeric(X['V4'], errors='coerce')

In [26]:
X.isna().sum()

V1    5
V2    5
V3    0
V4    0
V5    0
dtype: int64

In [27]:
y.isna().sum()

np.int64(0)

In [28]:
X[['V1', 'V2', 'V3', 'V4']] = X[['V1', 'V2', 'V3', 'V4']].astype('float')

In [46]:
impute_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('impute_scale', impute_scale, [0, 1]),
    ('scaler', StandardScaler(), [0, 1, 2, 3]),
    ('encoder', OrdinalEncoder(), [4])
])

In [47]:
X.shape

(748, 5)

In [48]:
X_transformed

array([[ 2.00000000e+00,  5.00000000e+01, -9.35028540e-01, ...,
         7.62334626e+00,  2.61563344e+00,  0.00000000e+00],
       [ 0.00000000e+00,  1.30000000e+01, -1.18230606e+00, ...,
         1.28273826e+00, -2.57880900e-01,  0.00000000e+00],
       [ 9.56258412e+00,  5.46433378e+00,             nan, ...,
         1.79684161e+00,  2.94705348e-02,  0.00000000e+00],
       ...,
       [ 2.30000000e+01,  3.00000000e+00,  1.66138547e+00, ...,
        -4.30939574e-01,  1.13782607e+00,  0.00000000e+00],
       [ 3.90000000e+01,  1.00000000e+00,  3.63960566e+00, ...,
        -7.73675141e-01,  1.93671355e-01,  0.00000000e+00],
       [ 7.20000000e+01,  1.00000000e+00,  7.71968482e+00, ...,
        -7.73675141e-01,  1.54832812e+00,  0.00000000e+00]],
      shape=(748, 7))

Question 2

In [41]:
enc = OrdinalEncoder()
y_encoded = enc.fit_transform(y.to_frame())
y_encoded_1d = y_encoded.ravel() 

In [42]:
preprocessor = ColumnTransformer([
    ('imputer', SimpleImputer(strategy='mean'), [0, 1]),
    ('scaler', StandardScaler(), [0, 1, 2, 3]),
    ('encoder', OrdinalEncoder(), [4])
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
])

X_transformed = pipeline.fit_transform(X, y_encoded)

In [43]:
y_encoded[:5]

array([[1.],
       [1.],
       [1.],
       [1.],
       [0.]])

In [44]:
X_transformed.T.shape

(7, 748)

In [45]:
estimator = LogisticRegression(random_state=1234)
selector = RFE(estimator, n_features_to_select=2)
selector.fit(X_transformed, y_encoded_1d)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
X_transformed

array([[ 2.00000000e+00,  5.00000000e+01, -9.35028540e-01, ...,
         7.62334626e+00,  2.61563344e+00,  0.00000000e+00],
       [ 0.00000000e+00,  1.30000000e+01, -1.18230606e+00, ...,
         1.28273826e+00, -2.57880900e-01,  0.00000000e+00],
       [ 9.56258412e+00,  5.46433378e+00,             nan, ...,
         1.79684161e+00,  2.94705348e-02,  0.00000000e+00],
       ...,
       [ 2.30000000e+01,  3.00000000e+00,  1.66138547e+00, ...,
        -4.30939574e-01,  1.13782607e+00,  0.00000000e+00],
       [ 3.90000000e+01,  1.00000000e+00,  3.63960566e+00, ...,
        -7.73675141e-01,  1.93671355e-01,  0.00000000e+00],
       [ 7.20000000e+01,  1.00000000e+00,  7.71968482e+00, ...,
        -7.73675141e-01,  1.54832812e+00,  0.00000000e+00]],
      shape=(748, 7))